In [1593]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRFRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

In [ ]:
df_matches = pd.read_csv("./data/gold_match.csv")
df_promo = pd.read_csv("./data/gold_match_context.csv")

df = pd.merge(df_matches, df_promo, on='match_id', how='left', suffixes=('', '_drop'))

df = df.filter(regex='^(?!.*_drop)')

In [1595]:
# Feature Engineering
df = df[df['is_home_match'] == True]
df['match_date'] = pd.to_datetime(df['match_date'])
df['day_number'] = df['match_date'].dt.dayofweek
df['hour'] = pd.to_datetime(df['kickoff_time_local']).dt.hour

C:\Users\nterh\AppData\Local\Temp\ipykernel_23436\1930472237.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['hour'] = pd.to_datetime(df['kickoff_time_local']).dt.hour


In [1596]:
df = df.sort_values('match_date')
df['days_since_last_home'] = df['match_date'].diff().dt.days
df['days_since_last_home'] = df['days_since_last_home'].fillna(14)

In [1597]:
cat_cols = ['away_team_code', 'weather_description', 'school_holiday_name']
num_cols = ['days_since_last_home', 'promo_tickets_total']
bool_cols = ['has_promotion', 'is_public_holiday', 'is_school_holiday_flanders', 'is_weekend']

In [1598]:
for col in bool_cols:
    df[col] = df[col].astype(int)

In [1599]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols),

        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('bool', SimpleImputer(strategy='constant', fill_value=0), bool_cols)
    ]
)

In [1600]:
X = df[cat_cols + num_cols + bool_cols]
y = df['tickets_scanned']

In [1601]:
ensemble = VotingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('xgb', XGBRFRegressor(n_estimators=100, learning_rate=0.01, max_depth=7, random_state=42))
    ]
)

In [1602]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ensemble)
])

In [1603]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [1604]:
model_pipeline.fit(X_train, y_train)
preds = model_pipeline.predict(X_test)

In [1605]:
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R2 Score: {r2:.2f}")

RMSE: 1858.60
MAE:  1575.17
R2 Score: 0.04


### without weather
1.5k tickets mae

### with weather
also 1.5k tickets mae

### without weather with days since last home game
1.5k tickets but more concetrated

In [1606]:
'''# Define the "search space"
param_grid = {
    'regressor__xgb__max_depth': [3, 5, 7],
    'regressor__xgb__learning_rate': [0.01, 0.1, 0.2],
    'regressor__rf__n_estimators': [100, 200]
}

# This will run the pipeline multiple times with different settings
grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, scoring='neg_mean_absolute_error')
grid_search.fit(X_train, y_train)

print(f"Best settings: {grid_search.best_params_}")'''

'# Define the "search space"\nparam_grid = {\n    \'regressor__xgb__max_depth\': [3, 5, 7],\n    \'regressor__xgb__learning_rate\': [0.01, 0.1, 0.2],\n    \'regressor__rf__n_estimators\': [100, 200]\n}\n\n# This will run the pipeline multiple times with different settings\ngrid_search = GridSearchCV(model_pipeline, param_grid, cv=3, scoring=\'neg_mean_absolute_error\')\ngrid_search.fit(X_train, y_train)\n\nprint(f"Best settings: {grid_search.best_params_}")'